In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG ecommerce_project")
spark.sql("USE SCHEMA sales")

In [0]:
%run ./00_transformations

In [0]:
agg = build_profit_aggregate(spark.table("enriched_orders"))
save_table(agg, "agg_profit", expected_min_rows=8129)

In [0]:
a = spark.table("agg_profit")
e = spark.table("enriched_orders")
print("agg rows            :", a.count())
print("profit agg/enriched :", a.agg(F.round(F.sum("total_profit"), 2)).first()[0], e.agg(F.round(F.sum("profit"), 2)).first()[0])
print("null customer_name  :", a.filter("customer_name IS NULL").count())
print("null category rows  :", a.filter("category IS NULL").count())
display(a.orderBy(F.desc("total_profit")).limit(5))

agg_total = a.agg(F.round(F.sum("total_profit"), 2)).first()[0]
src_total = e.agg(F.round(F.sum("profit"), 2)).first()[0]
assert abs(agg_total - src_total) < 0.5, f"Profit mismatch: {agg_total} vs {src_total}"
print("Reconciliation check passed.")